# 实验 0：Hopfield 检索怎样变形一条查询线？

**问题**：连续 Hopfield 检索是否压缩同一吸引域中的相邻查询，并在 A/B 吸引域分界附近放大微小差异？

这是一个**校准实验**：把 Pellegrino 与 Chadwick（NeurIPS 2025）的时间切片拉回度量方法，迁移到一个最小 Hopfield 系统。它不是论文复现，也不把这个直观现象当作新发现。

本实验固定：

- 两个存储模式 $A,B\in\{-1,+1\}^N$；
- 一维查询族 $x_0(\kappa)=(1-\kappa)A+\kappa B$；
- 连续 Hopfield 动力学 $\dot x=-x+W\tanh(gx)$；
- 欧式拉回度量 $G(t,\kappa)=\|\partial x_t/\partial\kappa\|_2^2$。

**预期**：吸引域内部的 $G$ 随时间缩小；分界附近的 $G$ 增大。真实分界由终点 overlap 独立确定，不由 $G$ 定义。


## 1. 导入工具

Google Colab 通常已经安装 JAX。第一次运行可能需要等待编译几十秒，CPU 即可。


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

jax.config.update("jax_enable_x64", True)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
})

print("JAX version:", jax.__version__)
print("device:", jax.devices()[0])


## 2. 构造两个记忆

$N$ 是状态空间的环境维度。这里的查询族只有一个参数 $\kappa$，所以它的内在维度是 1。


In [ ]:
N = 60
seed = 0
rng = np.random.default_rng(seed)

A = jnp.array(rng.choice([-1.0, 1.0], size=N))
B = jnp.array(rng.choice([-1.0, 1.0], size=N))

W = (jnp.outer(A, A) + jnp.outer(B, B)) / N
W = W - jnp.diag(jnp.diag(W))

gain = 4.0
pattern_overlap = float(A @ B / N)

print(f"N={N}, seed={seed}, overlap(A,B)={pattern_overlap:.3f}")


## 3. 连续 Hopfield 动力学

这里的 $x$ 是连续状态，因此可以对初始查询坐标 $\kappa$ 自动微分。RK4 只是数值积分器，不是研究变量。


In [ ]:
def vector_field(x):
    return -x + W @ jnp.tanh(gain * x)


dt = 0.02
n_steps = 300


def rk4_step(x, _):
    k1 = vector_field(x)
    k2 = vector_field(x + 0.5 * dt * k1)
    k3 = vector_field(x + 0.5 * dt * k2)
    k4 = vector_field(x + dt * k3)
    x_next = x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    return x_next, x_next


def trajectory(kappa):
    x0 = (1.0 - kappa) * A + kappa * B
    _, xs = jax.lax.scan(rk4_step, x0, None, length=n_steps)
    return jnp.concatenate([x0[None, :], xs], axis=0)


## 4. 拉回度量

在固定时刻 $t$，映射 $\kappa\mapsto x_t(\kappa)$ 把一条输入线嵌入 $\mathbb R^N$。JAX 计算它的切向量：

$$a_t(\kappa)=\frac{\partial x_t}{\partial\kappa}.$$

欧式状态空间上的拉回度量是：

$$G(t,\kappa)=a_t(\kappa)^\top a_t(\kappa).$$


In [ ]:
tangent_trajectory = jax.jacfwd(trajectory)


def metric_over_time(kappa):
    tangent = tangent_trajectory(kappa)
    return jnp.sum(tangent**2, axis=-1)


kappas = jnp.linspace(-0.3, 1.3, 121)

# 第一次执行会触发 JAX 编译。
trajectory_grid = jax.jit(jax.vmap(trajectory))(kappas)
G_grid = jax.jit(jax.vmap(metric_over_time))(kappas)

kappas_np = np.asarray(kappas)
time_np = np.arange(n_steps + 1) * dt
trajectory_np = np.asarray(trajectory_grid)
G_np = np.asarray(G_grid)

print("trajectory grid:", trajectory_np.shape)
print("metric grid:", G_np.shape)


## 5. 独立确定最终结局和分界

分界先由最终的 A/B overlap 差确定。随后再问 $G$ 的峰值是否靠近这个分界。


In [ ]:
overlap_A_t = np.asarray(trajectory_grid @ A / N)
overlap_B_t = np.asarray(trajectory_grid @ B / N)
final_margin = overlap_A_t[:, -1] - overlap_B_t[:, -1]
labels = np.where(final_margin > 0.0, 1, -1)

# overlap 差最接近 0 的位置：独立的分界估计。
boundary_from_overlap_idx = int(np.argmin(np.abs(final_margin)))
kappa_boundary_overlap = kappas_np[boundary_from_overlap_idx]

# 度量自己的峰值位置。
G_final = G_np[:, -1]
boundary_from_G_idx = int(np.argmax(G_final))
kappa_boundary_G = kappas_np[boundary_from_G_idx]

# 数值收敛检查。
final_speed = np.asarray(jax.vmap(lambda x: jnp.linalg.norm(vector_field(x)))(trajectory_grid[:, -1]))

expected_G0 = float(jnp.sum((B - A)**2))
print(f"G(0,kappa) 理论值 = ||B-A||² = {expected_G0:.3f}")
print(f"overlap 给出的分界  : kappa ≈ {kappa_boundary_overlap:.3f}")
print(f"G(T,kappa) 峰值位置: kappa ≈ {kappa_boundary_G:.3f}")
print(f"两种估计相差        : {abs(kappa_boundary_G-kappa_boundary_overlap):.3f}")
print(f"最大终点速度        : {final_speed.max():.3e}")


## 6. 主图

四幅图分别回答：何时出现拉伸、拉伸是否贴近分界、轨迹实际怎样分开、代表位置的度量怎样随时间变化。


In [ ]:
COLOR_A = "#3B82F6"
COLOR_B = "#F97316"
COLOR_BOUNDARY = "#111827"
COLOR_METRIC = "#D946EF"

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True)
ax_heat, ax_final, ax_overlap, ax_time = axes.ravel()

# A. time-slice pullback metric
log_G = np.log10(G_np + 1e-12)
vmin, vmax = np.percentile(log_G, [1, 99])
im = ax_heat.pcolormesh(
    kappas_np, time_np, log_G.T,
    shading="auto", cmap="magma", vmin=vmin, vmax=vmax,
)
ax_heat.axvline(kappa_boundary_overlap, color="white", ls="--", lw=1.5)
ax_heat.set(title="A  Dynamic warping", xlabel=r"query coordinate $\kappa$", ylabel="time")
fig.colorbar(im, ax=ax_heat, label=r"$\log_{10} G(t,\kappa)$")

# B. final metric and independently measured basin labels
ax_final.semilogy(kappas_np, G_final + 1e-12, color=COLOR_METRIC, lw=2.2)
ax_final.axvline(kappa_boundary_overlap, color=COLOR_BOUNDARY, ls="--", lw=1.5, label="overlap boundary")
ax_final.axvline(kappa_boundary_G, color="#10B981", ls=":", lw=2, label="metric peak")
ax_final.fill_between(kappas_np, 1e-12, G_final + 1e-12, where=labels > 0, color=COLOR_A, alpha=0.10)
ax_final.fill_between(kappas_np, 1e-12, G_final + 1e-12, where=labels < 0, color=COLOR_B, alpha=0.10)
ax_final.set(title="B  Readout-time sensitivity", xlabel=r"$\kappa$", ylabel=r"$G(T,\kappa)$")
ax_final.legend(frameon=False, fontsize=8)

# C. selected trajectories in memory-overlap coordinates
selected_kappas = [0.00, 0.44, 0.49, 0.50, 0.51, 0.56, 1.00]
cmap_paths = plt.get_cmap("coolwarm")
for value in selected_kappas:
    idx = int(np.argmin(np.abs(kappas_np - value)))
    color = cmap_paths((kappas_np[idx] - kappas_np.min()) / np.ptp(kappas_np))
    ax_overlap.plot(overlap_A_t[idx], overlap_B_t[idx], color=color, lw=1.8)
    ax_overlap.scatter(overlap_A_t[idx, 0], overlap_B_t[idx, 0], color=color, s=22, marker="o")
    ax_overlap.scatter(overlap_A_t[idx, -1], overlap_B_t[idx, -1], color=color, s=34, marker="x")
ax_overlap.scatter([1, 0], [0, 1], c=[COLOR_A, COLOR_B], s=90, marker="*", zorder=5)
ax_overlap.set(
    title="C  Retrieval trajectories",
    xlabel=r"overlap with $A$",
    ylabel=r"overlap with $B$",
)
ax_overlap.set_aspect("equal", adjustable="box")

# D. metric evolution at basin interior, near boundary, and boundary
probe_values = [(0.0, "inside A", COLOR_A), (0.48, "near boundary", "#A855F7"),
                (0.5, "boundary", COLOR_BOUNDARY), (1.0, "inside B", COLOR_B)]
for value, name, color in probe_values:
    idx = int(np.argmin(np.abs(kappas_np - value)))
    ax_time.semilogy(time_np, G_np[idx] / (G_np[idx, 0] + 1e-12), label=f"{name} (κ={kappas_np[idx]:.2f})", color=color, lw=2)
ax_time.axhline(1.0, color="gray", lw=1, ls=":")
ax_time.set(title="D  Compression versus stretching", xlabel="time", ylabel=r"$G(t,\kappa)/G(0,\kappa)$")
ax_time.legend(frameon=False, fontsize=8)

fig.suptitle("Continuous Hopfield retrieval warps a one-dimensional query manifold", fontsize=15, fontweight="bold")
plt.savefig("hopfield_pullback_experiment0.png", dpi=220, bbox_inches="tight")
plt.savefig("hopfield_pullback_experiment0.pdf", bbox_inches="tight")
plt.show()


## 7. 这张图允许说什么

如果结果符合预期，只能得出：

> 在这个对称的两记忆连续 Hopfield 系统中，有限时间动力学压缩吸引域内部的查询差异，并在 A/B 分界附近产生强烈拉伸。

它还不能说明：

- $G$ 比 overlap 更能预测未知错误；
- 该现象在高负载、相关模式或真实图片上仍成立；
- 度量峰值能够识别稳定伪吸引子；
- 这是此前没有发现的 Hopfield 现象。

下一关才是改变随机种子和模式相关性，并比较独立分界误差。


## 8. 运行后先回答三个问题

1. 为什么所有 $\kappa$ 在 $t=0$ 时具有相同的 $G$？
2. 为什么分界附近的 $G$ 增长，而 A、B 内部的 $G$ 下降？
3. 如果把观察时间 $T$ 增大，分界峰值会稳定、继续增长，还是消失？这对“用最终 $G$ 预测错误”有什么影响？


## 9. 方法来源与相邻研究

### 直接使用同一类方法

- Arthur Pellegrino & Angus Chadwick, **RNNs perform task computations by dynamically warping neural representations**, NeurIPS 2025. 论文从输入流形到时间变化的 RNN 状态构造 Jacobian 和 pullback metric；本 notebook 是对该方法的最小 Hopfield 迁移，而非论文复现。  
  https://papers.neurips.cc/paper_files/paper/2025/file/0572c069404875ccca76e822aaf48d50-Paper-Conference.pdf

### 研究 Hopfield 几何，但方法不同

- Kalmanje Krishnan & Tryphon Georgiou et al., **Hopfield Neural Network Flow: A Geometric Viewpoint**, IEEE TNNLS 2020。把连续 Hopfield 动力学解释为自然梯度流，并讨论随机情形的 Wasserstein 几何；不是查询流形的时间切片拉回度量。  
  https://arxiv.org/abs/1908.01270

- Tatiana Petrova, **Can Local Energy Geometry Predict Per-Pattern Retrieval Reliability in Dense Associative Memories?**, ICLR 2026 NFAM workshop。沿存储模式周围的随机切方向探测静态能量剖面，用局部非谐性预测检索可靠性；不是轨迹的动态拉回度量。  
  https://orbilu.uni.lu/bitstream/10993/68174/1/ICLR_Workshop_2-15.pdf

- David G. Clark, **Transient dynamics of associative memory models**, Physical Review E 2026。用 dynamical mean-field 和 transient-recovery curves 研究容量以上仍存在的慢区域和短暂检索；提供了不依赖流形度量的动态分析路线。  
  https://journals.aps.org/pre/abstract/10.1103/42y2-bsh1

- Chong Li et al., **Dynamic Manifold Hopfield Networks for Context-Dependent Associative Memory**, 2026 preprint。研究上下文如何重新塑造吸引子流形；它改变模型本身，与本 notebook 对固定模型做观察不同。  
  https://arxiv.org/abs/2506.01303
